# 00.2 — Devices: what `.to("mps")` does

**Question:** what happens when I move work to the Apple GPU, and which CUDA assumptions do not transfer?

**Interview one-liners**
- GPU calls are *asynchronous*: Python returns before the GPU finishes, so timing without `synchronize()` measures job hand-off, not compute.
- MPS is a different backend from CUDA: `float64` is unsupported, some ops fall back to CPU, numerics differ in the last bits.
- Unified memory: CPU and GPU share one RAM pool, so there is no PCIe copy like on a discrete GPU.

In [1]:
import sys, time, torch
sys.path.insert(0, '..')
from common import device_report
print(device_report())

torch 2.10.0 | mps_available=True | mps_built=True | cuda=False | selected=mps


## Experiment 1 — time a matmul three ways
Shapes: `(2048, 2048) @ (2048, 2048) -> (2048, 2048)`. FLOPs = 2·N³ ≈ 1.7e10.

**My prediction (written before running):** MPS without sync looks *faster* than reality, because the stopwatch stops when Python returns, not when the GPU finishes.

In [2]:
torch.manual_seed(0)
N = 2048
flops = 2 * N**3

def run(dev, sync):
    a = torch.randn(N, N, device=dev); b = torch.randn(N, N, device=dev)
    for _ in range(3): a @ b                      # warm-up
    if dev == 'mps': torch.mps.synchronize()      # drain the queue before starting the clock
    t = time.perf_counter()
    c = a @ b
    if sync and dev == 'mps': torch.mps.synchronize()   # wait for the GPU to actually finish
    return (time.perf_counter() - t) * 1000

results = {
    'cpu':         min(run('cpu', True)   for _ in range(5)),
    'mps NO sync': min(run('mps', False)  for _ in range(5)),
    'mps synced':  min(run('mps', True)   for _ in range(5)),
}
for name, ms in results.items():
    print(f'{name:12s} {ms:9.3f} ms   {flops / (ms/1000) / 1e12:8.3f} TFLOP/s')

cpu             11.684 ms      1.470 TFLOP/s
mps NO sync      0.037 ms    462.757 TFLOP/s
mps synced       4.320 ms      3.977 TFLOP/s


### Reading the output
- **No sync** implies ~500 TFLOP/s — impossible for an M3. It only timed the *hand-off*.
- **Synced** is the real number: ~2.7x faster than CPU here, not 100x.
- Lesson: one bad benchmark overstated speed by ~130x. Always synchronise. (`common/timing.py` does it for you.)

## Experiment 2 — `float64` on MPS

In [3]:
try:
    torch.ones(3, dtype=torch.float64, device='mps')
except Exception as e:
    print(type(e).__name__, ':', e)

TypeError : Cannot convert a MPS Tensor to float64 dtype as the MPS framework doesn't support float64. Please use float32 instead.


### Reading the output
A loud `TypeError`: MPS has no `float64`. Fine for ML (float32/bf16 are the norm), but it means CUDA/CPU code that uses float64 must be adapted.

## Challenge — predict first, then run
Shrink to `(64, 64) @ (64, 64)`. Will MPS still beat the CPU, or lose? Think: how much real work is there compared with the cost of handing a job to the GPU?

**My prediction:** _(write here)_

In [ ]:
# run AFTER you have written your prediction above
N = 64
flops = 2 * N**3
small = {'cpu': min(run('cpu', True) for _ in range(20)),
         'mps synced': min(run('mps', True) for _ in range(20))}
for name, ms in small.items():
    print(f'{name:12s} {ms:9.4f} ms')